# [실습 10-1] Keras CNN으로 CIFAR-10 분류 — 완전연결 대비 Before/After

| 항목 | 내용 |
|---|---|
| 예상 소요 시간 | 40~50분 (**GPU 권장** — Colab T4 기준 학습 셀 각 3~5분) |
| 본문 연계 | 10.2 CNN 구조 |
| 선수 실습 | [실습 9-2] (Keras 완전연결 MNIST — Before 모델의 원형) |
| 준비 | 부록 B.1·B.3 참고. 최초 실행 시 CIFAR-10 다운로드(약 170MB, 1회만) |

9장의 완전연결 신경망과 CNN을 **같은 데이터·같은 학습 설정**으로
겨루게 한다 — 바뀐 것은 구조뿐인데 결과가 달라지는 것이
"공간 구조를 보는 눈"의 효과다.

### [준비] 환경 설정 (저장소 전용)

In [ ]:
# 저장소 루트를 임포트 경로에 추가
# (Colab에서는 아래 두 줄의 주석을 해제하고 실행)
# !git clone https://github.com/tbgoodlife/ai-labs.git
# %cd ai-labs
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "utils").exists():
    ROOT = ROOT.parent          # ch10/에서 연 경우
sys.path.insert(0, str(ROOT))

import platform
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from utils import plot_style

plot_style.apply()              # 도해 스타일 킷 적용
print("Python", platform.python_version())
print("TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus[0].name if gpus else
      "없음 — Colab 메뉴 [런타임 유형 변경]에서 T4 선택")

### [셀 1] CIFAR-10 로드와 표본 확인 📖

In [ ]:
(x_tr, y_tr), (x_te, y_te) = \
    keras.datasets.cifar10.load_data()
x_tr, x_te = x_tr / 255.0, x_te / 255.0
y_tr, y_te = y_tr.ravel(), y_te.ravel()
CLASSES = ["비행기", "자동차", "새", "고양이", "사슴",
           "개", "개구리", "말", "배", "트럭"]
print("훈련", x_tr.shape, "/ 테스트", x_te.shape)

fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for d, ax in enumerate(axes):
    ax.imshow(x_tr[y_tr == d][0])
    ax.set_title(CLASSES[d], fontsize=8)
    ax.axis("off")
plt.show()

**핵심 포인트**
- 32×32 컬러 이미지 = 32×32×3 = **3,072차원** — MNIST(784)보다 4배 크고, 배경·색·자세 변화까지 있어 훨씬 어렵다.
- 픽셀값을 0~1로 정규화하는 것은 9장과 동일한 관례다.

기대 출력: 훈련 (50000, 32, 32, 3) / 테스트 (10000, 32, 32, 3)

### [셀 2] Before — 9장식 완전연결 모델 📖

In [ ]:
def build_dense():
    """9장 구조 그대로: 펼쳐서(Flatten) 완전연결."""
    return keras.Sequential([
        keras.layers.Input((32, 32, 3)),
        keras.layers.Flatten(),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])

dense = build_dense()
dense.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
print(f"파라미터 수: {dense.count_params():,}")
hist_d = dense.fit(x_tr, y_tr, epochs=10,
                   batch_size=128,
                   validation_split=0.1, verbose=2)

**핵심 포인트**
- `Flatten`이 문제의 근원이다 — 이미지를 한 줄로 펴는 순간 "옆 픽셀"이라는 공간 정보가 사라진다(10.1.1).
- 첫 층에만 3,072×512 ≈ **157만 파라미터** — 본문 10.1.1의 파라미터 폭발 계산이 여기서 실물로 나온다.

실패 시 대처: 학습이 매우 느리면 GPU 연결을 확인한다([준비] 셀 안내).

### [셀 3] After — CNN 모델 📖

In [ ]:
def build_cnn():
    """합성곱-풀링 2스택 + 분류층(10.2 표준 구조)."""
    return keras.Sequential([
        keras.layers.Input((32, 32, 3)),
        keras.layers.Conv2D(32, 3, padding="same",
                            activation="relu"),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, padding="same",
                            activation="relu"),
        keras.layers.MaxPooling2D(),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(10, activation="softmax"),
    ])

cnn = build_cnn()
cnn.compile(optimizer="adam",
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
cnn.summary()

**핵심 포인트**
- 첫 합성곱 층의 파라미터는 3×3×3×32+32 = **896개** — 본문 10.2.1의 파라미터 수 공식 손계산과 같은 숫자다. 필터 하나를 이미지 전체가 **공유**하기 때문에 완전연결보다 압도적으로 적다.
- 구조 읽는 법: Conv(특징 감지) → MaxPool(요약·축소)을 두 번 반복해 "저수준 → 고수준" 특징 계층을 만든다(10.2.3).

### [셀 4] 학습과 Before/After 비교 📖

In [ ]:
hist_c = cnn.fit(x_tr, y_tr, epochs=10,
                 batch_size=128,
                 validation_split=0.1, verbose=2)

_, acc_d = dense.evaluate(x_te, y_te, verbose=0)
_, acc_c = cnn.evaluate(x_te, y_te, verbose=0)
print(f"{'모델':<14}{'파라미터':>12}{'테스트 정확도':>10}")
print(f"{'완전연결(9장식)':<14}"
      f"{dense.count_params():>14,}{acc_d:>12.3f}")
print(f"{'CNN':<14}{cnn.count_params():>14,}"
      f"{acc_c:>12.3f}")

**핵심 포인트**
- 같은 에포크에서 CNN이 완전연결을 큰 폭으로 앞선다(기대: 약 45~50% vs 70% 내외) — 게다가 **파라미터는 더 적다**(가중치 공유의 힘).
- 정확도 수치는 초기화에 따라 ±2%p 흔들릴 수 있다 — 순위가 뒤집히지 않는 것이 관찰 포인트다.

실패 시 대처: 두 모델의 순위가 뒤집혔다면 에포크 수·정규화 셀이 수정되지 않았는지 확인한다.

### [보조 1] 학습 곡선 겹쳐 보기

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(hist_d.history["val_accuracy"], "--",
        color="#8c8c8c", lw=1.2, marker="s", ms=3,
        label="완전연결(검증)")
ax.plot(hist_c.history["val_accuracy"], "-",
        color="black", lw=1.2, marker="o", ms=3,
        label="CNN(검증)")
ax.set_xlabel("에포크")
ax.set_ylabel("정확도")
ax.legend()
plt.show()

### [보조 2] CNN은 무엇을 보고 있는가 — 특징 맵 엿보기

In [ ]:
# 첫 합성곱 층의 출력(특징 맵) 8장을 시각화한다
probe = keras.Model(cnn.inputs,
                    cnn.layers[0].output)
fmap = probe.predict(x_te[:1], verbose=0)[0]

fig, axes = plt.subplots(1, 9, figsize=(10, 1.4))
axes[0].imshow(x_te[0])
axes[0].set_title("입력", fontsize=8)
for i, ax in enumerate(axes[1:]):
    ax.imshow(fmap[:, :, i], cmap="gray")
    ax.set_title(f"필터 {i}", fontsize=8)
for ax in axes:
    ax.axis("off")
plt.show()
# 어떤 필터는 윤곽선에, 어떤 필터는 색 덩어리에
# 반응한다 — 심화 박스 "CNN은 무엇을 보고 있는가" 연동

### [심화 1] 구조 변형 실험 (연습문제 심화 연계)

In [ ]:
# TODO: 아래 손잡이를 하나씩 바꿔 [셀 4] 비교표를
#       갱신하고, 효과를 과적합 관점(6장)으로 해석하자.
# 1) Conv-Pool 스택을 3개로 늘리기
# 2) 필터 수 32→64→128로 두 배씩
# 3) Dropout 0.5 제거 시 훈련/검증 곡선 간격 관찰
def build_variant(stacks=2, base=32, dropout=0.5):
    layers = [keras.layers.Input((32, 32, 3))]
    for i in range(stacks):
        layers += [
            keras.layers.Conv2D(base * 2**i, 3,
                padding="same", activation="relu"),
            keras.layers.MaxPooling2D()]
    layers += [keras.layers.Flatten(),
               keras.layers.Dense(128,
                   activation="relu")]
    if dropout:
        layers.append(keras.layers.Dropout(dropout))
    layers.append(keras.layers.Dense(10,
        activation="softmax"))
    return keras.Sequential(layers)

print(build_variant(3, 32).count_params())

---
## 마무리

- 완전연결의 한계는 공간 구조의 소실과 파라미터 폭발 — CNN은 **국소 연결·가중치 공유**로 둘 다 해결한다.
- Conv(감지)-Pool(요약) 스택이 특징의 계층을 만든다 — [보조 2]에서 첫 층이 실제로 보는 것을 확인했다.
- 같은 데이터·같은 설정에서 구조만으로 정확도가 갈렸다 — "구조가 곧 사전 지식"이라는 딥러닝 설계 철학의 첫 체험.

**연습문제 연계**: [응용] 합성곱 출력 크기·파라미터 수 계산은 [셀 3] `summary()`로 자가 검증, [심화] 구조 변형은 [심화 1]에서 수행한다.

**다음 실습**: [실습 10-2] 전이학습 (`lab-10-02_transfer-learning.ipynb`)